# T1. [연습] MS MARCO v1.1 건강 데이터 전처리

본 작업(v2.1) 전에 **작은 v1.1로 전체 흐름을 익히는 연습용**입니다.

**이 노트북이 하는 일**
1. MS MARCO v1.1을 불러옵니다 (질문 약 9만 개, 질문마다 Bing이 찾은 웹 문단 약 8~10개).
2. 문단 URL을 보고 **건강 질문**만 골라냅니다.
3. 검색 대상이 될 **문단 모음**(`corpus.jsonl`)을 만듭니다.
4. 검색을 채점할 **평가 질문**(`eval.jsonl`)을 고릅니다.

다음 노트북 `T2_v11_embedding (test).ipynb`가 이 파일들로 임베딩과 검색을 합니다.

| 결과 파일 | 내용 |
|---|---|
| `corpus.jsonl` | 문단 약 15만 개. 한 줄 = `{"pid", "text", "url", "src"}` |
| `eval.jsonl` | 평가 질문 550개 = 건강 질문 500 + 분야 밖 질문 50 |

**v1.1의 한계:** 답 없는 질문(`No Answer Present.`)이 없어서 "모르면 거절" 평가는 할 수 없습니다. 그건 v2.1 본 작업에서 합니다.

런타임은 CPU로도 충분합니다. 위에서부터 순서대로 실행하세요.

## 1. 저장 위치

In [1]:
# 드라이브 연결 (안 되면 서버 안에 저장 → 서버를 끄면 사라짐)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/rag_msmarco/v11_test'
except Exception as e:
    import os
    if os.path.exists('/content'):            # Colab인데 드라이브만 안 됨
        print('드라이브 연결 실패 → 서버 안에 저장합니다:', e)
        SAVE_DIR = '/content/rag_msmarco/v11_test'
    else:                                     # 내 컴퓨터에서 실행
        print('Colab이 아님 → 노트북 옆 폴더에 저장합니다')
        SAVE_DIR = os.path.abspath('v11_test_data')
print('저장 위치:', SAVE_DIR)

Mounted at /content/drive
저장 위치: /content/drive/MyDrive/rag_msmarco/v11_test


## 2. 설정

숫자를 바꾸면 데이터 크기가 바뀝니다. 처음에는 그대로 두세요.

In [2]:
import os, re, json, random, time
from collections import Counter
import numpy as np

def save_jsonl(rows, path):
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f]

SEED            = 42      # 랜덤 고정 → 누가 돌려도 같은 결과
MIN_HEALTH_HITS = 2       # 질문의 문단 중 건강 사이트 문단이 이 개수 이상이면 건강 질문
N_DISTRACTOR_Q  = 3000    # 방해 문단을 가져올 일반(건강 아님) 질문 수
N_EVAL_HEALTH   = 500     # 평가: 건강 질문
N_EVAL_OUT      = 50      # 평가: 분야 밖 질문 (나중에 "건강 질문 아니면 거절" 확인용)

os.makedirs(SAVE_DIR, exist_ok=True)

## 3. 건강 사이트 목록

v2.1 dev 문단 100만 개의 출처(URL)를 세어 보고, 많이 나오는 의료·건강 사이트를 고른 목록입니다.

In [3]:
from urllib.parse import urlparse

HEALTH_SITES = [
    'webmd.com', 'mayoclinic.org', 'medicinenet.com', 'medical-dictionary.thefreedictionary.com',
    'verywell.com', 'healthline.com', 'drugs.com', 'livestrong.com', 'healthtap.com', 'cdc.gov',
    'rightdiagnosis.com', 'emedicinehealth.com', 'medscape.com', 'everydayhealth.com',
    'medlineplus.gov', 'nhs.uk', 'rxlist.com', 'sharecare.com', 'medicalnewstoday.com',
    'kidshealth.org', 'nih.gov', 'patient.info', 'labtestsonline.org', 'aafp.org',
    'clevelandclinic.org', 'hopkinsmedicine.org', 'healthgrades.com', 'familydoctor.org',
    'merckmanuals.com', 'uptodate.com', 'health.harvard.edu', 'babycenter.com', 'psychologytoday.com',
]

def host(url):
    h = urlparse(url).netloc.lower()
    return h[4:] if h.startswith('www.') else h

def is_health(url):
    h = host(url)
    return any(h == s or h.endswith('.' + s) for s in HEALTH_SITES)

# 확인: True True False 가 나오면 정상
print(is_health('http://www.webmd.com/a'), is_health('https://my.clevelandclinic.org/b'), is_health('https://en.wikipedia.org/c'))

True True False


## 4. 데이터 불러오기

Hugging Face에서 자동으로 받습니다. 파일을 따로 받을 필요가 없습니다.

In [4]:
from datasets import load_dataset

ds = load_dataset('microsoft/ms_marco', 'v1.1')
dev, train = ds['validation'], ds['train']
print(ds)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/9.48k [00:00<?, ?B/s]

v1.1/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.4MB            

v1.1/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

v1.1/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  175MB            

...

v1.1/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating validation split:   0%|          | 0/10047 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/82326 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/9650 [00:00<?, ? examples/s]

DatasetDict({
    validation: Dataset({
        features: ['answers', 'passages', 'query', 'query_id', 'query_type', 'wellFormedAnswers'],
        num_rows: 10047
    })
    train: Dataset({
        features: ['answers', 'passages', 'query', 'query_id', 'query_type', 'wellFormedAnswers'],
        num_rows: 82326
    })
    test: Dataset({
        features: ['answers', 'passages', 'query', 'query_id', 'query_type', 'wellFormedAnswers'],
        num_rows: 9650
    })
})


### 데이터 한 줄 들여다보기

질문 1개에 `passages`(문단 목록)가 붙어 있고, 각 문단에 `is_selected`(1 = 정답이 들어 있는 문단)와 `url`이 있습니다.

In [5]:
ex = dev[0]
print('질문:', ex['query'])
print('질문 종류:', ex['query_type'])
print('정답:', ex['answers'])
print('문단 수:', len(ex['passages']['passage_text']))
for text, sel, url in zip(ex['passages']['passage_text'][:3], ex['passages']['is_selected'][:3], ex['passages']['url'][:3]):
    print(f'\n  [정답 문단={sel}] {url}\n  {text[:200]}')

질문: walgreens store sales average
질문 종류: numeric
정답: ['Approximately $15,000 per year.']
문단 수: 6

  [정답 문단=1] http://www.indeed.com/cmp/Walgreens/salaries
  The average Walgreens salary ranges from approximately $15,000 per year for Customer Service Associate / Cashier to $179,900 per year for District Manager. Average Walgreens hourly pay ranges from app

  [정답 문단=0] http://www.answers.com/Q/What_is_the_average_gross_sales_volume_of_a_single_Walgreen's_Store
  The average revenue in 2011 of a Starbuck Store was $1,078,000, up  from $1,011,000 in 2010.    The average ticket (total purchase) at domestic Starbuck stores in  No … vember 2007 was reported at $6.

  [정답 문단=0] http://news.walgreens.com/fact-sheets/frequently-asked-questions.htm
  In fiscal 2014, Walgreens opened a total of 184 new locations and acquired 84 locations, for a net decrease of 273 after relocations and closings. How big are your stores? The average size for a typic


## 5. 정리 함수 + 건강 질문 고르기

- 질문 앞의 `". "` 같은 쓰레기 문자를 지우고, 문단의 줄바꿈·연속 공백을 정리합니다.
- 질문마다 **건강 사이트 문단이 몇 개인지**(`hits`) 셉니다.

In [6]:
def clean_query(q):
    q = ' '.join(q.split())
    return re.sub(r'^[\s.,;:!?\-]+', '', q)

def clean_text(t):
    return ' '.join(t.split())

def clean_list(v):
    return [clean_text(a) for a in v if a and a.strip()]

def count_hits(split):
    hits = []
    for b in split.select_columns(['passages']).iter(batch_size=5000):
        hits += [sum(is_health(u) for u in p['url']) for p in b['passages']]
    return np.array(hits)

def iter_rows(split, idx):
    """idx 순서대로 한 행씩 dict로 돌려줍니다."""
    for b in split.select(list(map(int, idx))).iter(batch_size=1000):
        for j in range(len(b['query'])):
            yield {k: b[k][j] for k in b}

dev_hits, train_hits = count_hits(dev), count_hits(train)
print('dev   건강 질문:', (dev_hits >= MIN_HEALTH_HITS).sum(), '/', len(dev))
print('train 건강 질문:', (train_hits >= MIN_HEALTH_HITS).sum(), '/', len(train))

dev   건강 질문: 1716 / 10047
train 건강 질문: 14091 / 82326


## 6. 문단 모음 만들기

세 곳에서 문단을 모읍니다. 내용이 똑같은 문단은 하나만 남기고, `pid`(문단 번호)를 0부터 붙입니다.

| `src` | 출처 | 왜 넣나 |
|---|---|---|
| `dev_health` | dev 건강 질문의 문단 | 평가 질문의 **정답 문단**이 여기 있음 |
| `train_health` | train 건강 질문의 문단 전부 | 건강 문서를 늘려서 검색을 어렵게 (실제 서비스처럼) |
| `dev_other` | dev 일반 질문 일부의 문단 | **방해 문단** (건강과 상관없는 문서) |

`is_selected=1`인 문단의 pid를 질문의 `gold_pids`(정답 문단 번호)로 저장합니다. 나중에 검색 채점에 씁니다.

In [7]:
rng = random.Random(SEED)

dev_health_idx   = np.where(dev_hits >= MIN_HEALTH_HITS)[0]
train_health_idx = np.where(train_hits >= MIN_HEALTH_HITS)[0]
dev_other_idx    = np.where(dev_hits == 0)[0].tolist()
rng.shuffle(dev_other_idx)
distractor_idx = sorted(dev_other_idx[:N_DISTRACTOR_Q])
out_pool_idx   = dev_other_idx[N_DISTRACTOR_Q:]      # 분야 밖 평가 질문 후보 (이 질문들의 문단은 안 넣음)

corpus, pid_of, n_dup = [], {}, 0

def add_passages(p, src):
    """문단을 모음에 넣고, 정답 문단(is_selected=1)의 pid 목록을 돌려줍니다."""
    global n_dup
    gold = []
    for text, sel, url in zip(p['passage_text'], p['is_selected'], p['url']):
        text = clean_text(text)
        if not text:
            continue
        if text in pid_of:
            n_dup += 1
        else:
            pid_of[text] = len(corpus)
            corpus.append({'pid': len(corpus), 'text': text, 'url': url, 'src': src})
        if sel == 1 and pid_of[text] not in gold:
            gold.append(pid_of[text])
    return gold

def to_question(row, gold, domain):
    answers = clean_list(row['answers'])
    return {
        'qid': int(row['query_id']),
        'query': clean_query(row['query']),
        'gold_pids': gold,
        'answers': answers,
        'answerable': bool(answers) and len(gold) > 0,
        'query_type': row['query_type'],
        'domain': domain,
    }

dev_health_q = [to_question(r, add_passages(r['passages'], 'dev_health'), 'health')
                for r in iter_rows(dev, dev_health_idx)]
for r in iter_rows(train, train_health_idx):
    add_passages(r['passages'], 'train_health')
for r in iter_rows(dev, distractor_idx):
    add_passages(r['passages'], 'dev_other')

print(f'문단 {len(corpus):,}개 (중복 {n_dup:,}개 제거)')
print('출처별:', Counter(c['src'] for c in corpus))

문단 142,605개 (중복 14,718개 제거)
출처별: Counter({'train_health': 104494, 'dev_other': 24087, 'dev_health': 14024})


## 7. 평가 질문 고르기

- 건강 질문 500개: 정답 문장이 있고, 정답 문단도 1개 이상인 질문
- 분야 밖 질문 50개: 건강 사이트 문단이 하나도 없는 질문. 이 질문들의 문단은 모음에 넣지 않았습니다.

In [8]:
health_pool = [q for q in dev_health_q if q['answerable']]
print('건강 평가 후보:', len(health_pool))

rng = random.Random(SEED)
out_q = [to_question(r, [], 'out') for r in iter_rows(dev, rng.sample(out_pool_idx, N_EVAL_OUT))]
for q in out_q:
    q['answerable'] = False          # 분야 밖 → 나중에 생성 단계에서 거절해야 정답

eval_set = rng.sample(health_pool, N_EVAL_HEALTH) + out_q
print('평가 질문:', len(eval_set), Counter(q['domain'] for q in eval_set))
print('질문 종류:', Counter(q['query_type'] for q in eval_set))

건강 평가 후보: 1676
평가 질문: 550 Counter({'health': 500, 'out': 50})
질문 종류: Counter({'description': 335, 'numeric': 101, 'entity': 95, 'location': 15, 'person': 4})


## 8. 저장 + 확인

In [9]:
save_jsonl(corpus,   f'{SAVE_DIR}/corpus.jsonl')
save_jsonl(eval_set, f'{SAVE_DIR}/eval.jsonl')
with open(f'{SAVE_DIR}/meta.json', 'w') as f:
    json.dump({'source': 'huggingface microsoft/ms_marco v1.1', 'seed': SEED,
               'min_health_hits': MIN_HEALTH_HITS, 'n_passages': len(corpus),
               'src_counts': Counter(c['src'] for c in corpus),
               'n_dev_health_q': len(dev_health_q), 'n_train_health_q': len(train_health_idx),
               'n_eval': {'health': N_EVAL_HEALTH, 'out_of_domain': N_EVAL_OUT}}, f, indent=2)

for name in ['corpus.jsonl', 'eval.jsonl', 'meta.json']:
    print(name, f'{os.path.getsize(f"{SAVE_DIR}/{name}") / 1e6:.1f} MB')

# 확인 1: pid가 줄 번호와 같은지 / 확인 2: 정답 pid가 범위 안인지
assert all(c['pid'] == i for i, c in enumerate(corpus)), 'pid가 줄 번호와 다름'
assert all(0 <= p < len(corpus) for q in eval_set for p in q['gold_pids']), '범위 밖 정답 pid'
print('검사 통과')

# 확인 3: 눈으로 보기. 정답 문단 안에 정답 내용이 보이면 연결이 맞은 것
for q in random.Random(0).sample(health_pool, 3):
    print('\n질문:', q['query'])
    print('정답:', q['answers'][0][:200])
    print('정답 문단:', corpus[q['gold_pids'][0]]['text'][:300])

corpus.jsonl 78.9 MB
eval.jsonl 0.2 MB
meta.json 0.0 MB
검사 통과

질문: lowest viable body temperature
정답: 35 C
정답 문단: Definition. Hypothermia is a medical emergency that occurs when your body loses heat faster than it can produce heat, causing a dangerously low body temperature. Normal body temperature is around 98.6 F (37 C). Hypothermia (hi-poe-THUR-me-uh) occurs as your body temperature passes below 95 F (35 C).

질문: can you prevent appendicitis pains
정답: Yes
정답 문단: How to Prevent Appendicitis: A Summary. Common symptoms of appendicitis include: 1 Pain in the abdomen, first around the belly button, then moving to the lower right area. 2 Loss of appetite. 3 Nausea. 4 Vomiting. 5 Constipation or diarrhea. 6 Inability to pass gas. 7 Low fever that begins after oth

질문: what is a wild carrot
정답: It is a biennial plant.
정답 문단: Wild carrot is a biennial plant. During the first year only leaves are formed. The second year the plant forms a single 1m long erect stem with small white flowers. Th